# Triton Inference Demo

Minimalistic examples showing how to invoke all Triton models via **two protocols**:
- **gRPC** - Binary protobuf, fastest for all tensor sizes
- **REST (Binary)** - HTTP with binary tensor encoding, good balance of performance and HTTP compatibility

**Models covered:**
- YOLOv8n (Object Detection) - Video/Image
- BERT (Text Classification) - Text
- Whisper (Audio Transcription) - Audio
- SmolLM-135M (Text Generation) - Text

---
## Note on REST (JSON) Protocol

REST JSON encoding (`binary_data=False`) is also supported but has a known issue in Jupyter notebooks where it fails with JSON parsing errors when run after other inference calls or library imports in the same kernel session. This appears to be a state corruption issue in the `tritonclient.http` library within Jupyter.

**REST JSON works reliably:**
- In standalone Python scripts (benchmarks work fine)
- In fresh Jupyter kernel with no other cells executed
- See the sanity check cells (3-4) for working examples

**For production use:** gRPC or REST Binary are recommended anyway due to better performance.

---
## Prerequisites

Install the client dependencies before running this notebook:

```bash
pip install -r docker/requirements-client.txt
```

This installs:
- `tritonclient[all]` - Triton gRPC and HTTP clients
- `opencv-python-headless` - Video/image processing
- `transformers` - Tokenizers and processors
- `librosa` - Audio processing
- `numpy`, `torch`, etc.

## Setup

In [ ]:
import os
import time
import numpy as np

# =============================================================================
# Configuration - Update these for your environment
# =============================================================================
os.environ["TRITON_GRPC_URL"] = "triton-inference-server-proxy.domino-inference-dev.svc.cluster.local:50051"
os.environ["TRITON_REST_URL"] = "http://triton-inference-server-proxy.domino-inference-dev.svc.cluster.local:8080"

GRPC_URL = os.environ.get("TRITON_GRPC_URL", "localhost:50051")
REST_URL = os.environ.get("TRITON_REST_URL", "http://localhost:8080")

# =============================================================================
# Authentication
# =============================================================================
API_KEY = os.environ.get("DOMINO_USER_API_KEY", "")

if API_KEY:
    grpc_headers = {"x-domino-api-key": API_KEY}
    http_headers = {"X-Domino-Api-Key": API_KEY}
else:
    grpc_headers = None
    http_headers = None

print(f"gRPC URL: {GRPC_URL}")
print(f"REST URL: {REST_URL}")
print(f"Auth: {'API Key configured' if API_KEY else 'Disabled'}")

In [ ]:
# =============================================================================
# SANITY CHECK: Minimal REST JSON test (run after fresh kernel restart)
# =============================================================================
# This is the minimal code that worked from the command line.
# If this fails, there's a Jupyter kernel or environment issue.

import tritonclient.http as httpclient
import numpy as np

# Small test tensor
test_data = np.array([[1.0, 2.0, 3.0]], dtype=np.float32)

url = REST_URL.replace("http://", "").replace("https://", "")
client = httpclient.InferenceServerClient(url=url)

# Test with BERT (small INT64 payload)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
encoded = tokenizer(["Test sentence"], padding="max_length", truncation=True, max_length=128, return_tensors="np")

input_ids = encoded["input_ids"].astype(np.int64)
attention_mask = encoded["attention_mask"].astype(np.int64)

print(f"input_ids shape: {input_ids.shape}, dtype: {input_ids.dtype}")
print(f"Testing REST JSON with BERT...")

input_ids_tensor = httpclient.InferInput("input_ids", list(input_ids.shape), "INT64")
input_ids_tensor.set_data_from_numpy(input_ids, binary_data=False)
attention_mask_tensor = httpclient.InferInput("attention_mask", list(attention_mask.shape), "INT64")
attention_mask_tensor.set_data_from_numpy(attention_mask, binary_data=False)
output_tensor = httpclient.InferRequestedOutput("logits", binary_data=False)

response = client.infer(
    model_name="bert-base-uncased",
    inputs=[input_ids_tensor, attention_mask_tensor],
    outputs=[output_tensor],
    headers=http_headers,
)
logits = response.as_numpy("logits")
print(f"✓ REST JSON works! Output shape: {logits.shape}")

In [ ]:
# =============================================================================
# SANITY CHECK: YOLOv8 ALL THREE PROTOCOLS (self-contained)
# =============================================================================
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient
import cv2

# Load and preprocess one frame (self-contained, no helper functions)
video_path = "../samples/video.avi"
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

h, w = frame.shape[:2]
scale = min(640 / h, 640 / w)
new_w, new_h = int(w * scale), int(h * scale)
resized = cv2.resize(frame, (new_w, new_h))
img = np.full((640, 640, 3), 114, dtype=np.uint8)
pad_h, pad_w = (640 - new_h) // 2, (640 - new_w) // 2
img[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
base_tensor = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
base_tensor = np.transpose(base_tensor, (2, 0, 1))

print(f"Base tensor shape: {base_tensor.shape}")

# =============================================================================
# 1. REST (JSON) - FIRST
# =============================================================================
tensor1 = base_tensor[np.newaxis, ...].astype(np.float32).copy()
url = REST_URL.replace("http://", "").replace("https://", "")
client1 = httpclient.InferenceServerClient(url=url)
inp1 = httpclient.InferInput("images", list(tensor1.shape), "FP32")
inp1.set_data_from_numpy(tensor1, binary_data=False)
out1 = httpclient.InferRequestedOutput("output0", binary_data=False)

import time
start = time.time()
resp1 = client1.infer("yolov8n", [inp1], outputs=[out1], headers=http_headers)
t1 = (time.time() - start) * 1000
print(f"✓ REST (JSON):   {t1:8.2f} ms | Output: {resp1.as_numpy('output0').shape}")

# =============================================================================
# 2. gRPC
# =============================================================================
tensor2 = base_tensor[np.newaxis, ...].astype(np.float32).copy()
client2 = grpcclient.InferenceServerClient(url=GRPC_URL)
inp2 = grpcclient.InferInput("images", list(tensor2.shape), "FP32")
inp2.set_data_from_numpy(tensor2)
out2 = grpcclient.InferRequestedOutput("output0")

start = time.time()
resp2 = client2.infer("yolov8n", [inp2], outputs=[out2], headers=grpc_headers)
t2 = (time.time() - start) * 1000
client2.close()
print(f"✓ gRPC:          {t2:8.2f} ms | Output: {resp2.as_numpy('output0').shape}")

# =============================================================================
# 3. REST (Binary)
# =============================================================================
tensor3 = base_tensor[np.newaxis, ...].astype(np.float32).copy()
client3 = httpclient.InferenceServerClient(url=url)
inp3 = httpclient.InferInput("images", list(tensor3.shape), "FP32")
inp3.set_data_from_numpy(tensor3, binary_data=True)
out3 = httpclient.InferRequestedOutput("output0", binary_data=True)

start = time.time()
resp3 = client3.infer("yolov8n", [inp3], outputs=[out3], headers=http_headers)
t3 = (time.time() - start) * 1000
print(f"✓ REST (Binary): {t3:8.2f} ms | Output: {resp3.as_numpy('output0').shape}")

print("\n✓ All three protocols work!")

---
## 1. YOLOv8n (Object Detection)

Input: Image tensor `[batch, 3, 640, 640]` (float32, normalized 0-1)  
Output: Detections `[batch, 84, 8400]` (boxes + class scores)

In [ ]:
import cv2

def preprocess_frame(frame):
    """Preprocess frame for YOLOv8n: resize, pad, normalize, transpose."""
    h, w = frame.shape[:2]
    scale = min(640 / h, 640 / w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(frame, (new_w, new_h))
    
    # Pad to 640x640
    img = np.full((640, 640, 3), 114, dtype=np.uint8)
    pad_h, pad_w = (640 - new_h) // 2, (640 - new_w) // 2
    img[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
    
    # BGR -> RGB, normalize, HWC -> CHW
    tensor = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = np.transpose(tensor, (2, 0, 1))
    return tensor

def extract_frames(video_path, max_frames=5):
    """Extract frames from video."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames

### YOLOv8n - gRPC and REST Binary

In [ ]:
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient
import cv2

# Load and preprocess one frame (self-contained, no helper functions)
video_path = "../samples/video.avi"
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

h, w = frame.shape[:2]
scale = min(640 / h, 640 / w)
new_w, new_h = int(w * scale), int(h * scale)
resized = cv2.resize(frame, (new_w, new_h))
img = np.full((640, 640, 3), 114, dtype=np.uint8)
pad_h, pad_w = (640 - new_h) // 2, (640 - new_w) // 2
img[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
base_tensor = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
base_tensor = np.transpose(base_tensor, (2, 0, 1))

print(f"Input shape: [1, {base_tensor.shape[0]}, {base_tensor.shape[1]}, {base_tensor.shape[2]}]")

url = REST_URL.replace("http://", "").replace("https://", "")

# =============================================================================
# 1. gRPC Protocol
# =============================================================================
tensor_grpc = base_tensor[np.newaxis, ...].astype(np.float32).copy()
client = grpcclient.InferenceServerClient(url=GRPC_URL)
inp = grpcclient.InferInput("images", list(tensor_grpc.shape), "FP32")
inp.set_data_from_numpy(tensor_grpc)
out = grpcclient.InferRequestedOutput("output0")

start = time.time()
resp = client.infer("yolov8n", [inp], outputs=[out], headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
output_grpc = resp.as_numpy("output0")
client.close()
print(f"✓ gRPC:          {t_grpc:8.2f} ms | Output: {output_grpc.shape}")

# =============================================================================
# 2. REST (Binary) Protocol
# =============================================================================
tensor_binary = base_tensor[np.newaxis, ...].astype(np.float32).copy()
client = httpclient.InferenceServerClient(url=url)
inp = httpclient.InferInput("images", list(tensor_binary.shape), "FP32")
inp.set_data_from_numpy(tensor_binary, binary_data=True)
out = httpclient.InferRequestedOutput("output0", binary_data=True)

start = time.time()
resp = client.infer("yolov8n", [inp], outputs=[out], headers=http_headers)
t_binary = (time.time() - start) * 1000
output_binary = resp.as_numpy("output0")
print(f"✓ REST (Binary): {t_binary:8.2f} ms | Output: {output_binary.shape}")

# Verify outputs match
print(f"\nOutputs match (gRPC == REST Binary): {np.allclose(output_grpc, output_binary)}")

# Save for decoding cell
output = output_grpc

### Decoding YOLOv8n Output

The raw output `[batch, 84, 8400]` contains:
- **84** = 4 box coords (cx, cy, w, h) + 80 COCO class scores
- **8400** = detection anchor points

Post-processing extracts bounding boxes with confidence > threshold:

In [ ]:
# COCO class names (80 classes)
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
    "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
    "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball",
    "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket",
    "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
    "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator",
    "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

def decode_yolo_output(output, conf_threshold=0.25):
    """
    Decode YOLOv8 output tensor to list of detections.
    
    Args:
        output: Raw output [batch, 84, 8400]
        conf_threshold: Minimum confidence to include detection
    
    Returns:
        List of dicts with class, confidence, and bbox (cx, cy, w, h)
    """
    detections = []
    
    # Get first batch item, transpose to [8400, 84]
    pred = output[0].T  # [8400, 84]
    
    # Split: first 4 = box coords, rest = class scores
    boxes = pred[:, :4]        # [8400, 4] - cx, cy, w, h
    class_scores = pred[:, 4:] # [8400, 80]
    
    # Get best class per detection
    class_ids = np.argmax(class_scores, axis=1)
    confidences = np.max(class_scores, axis=1)
    
    # Filter by confidence
    mask = confidences >= conf_threshold
    
    for i in np.where(mask)[0]:
        detections.append({
            "class": COCO_CLASSES[class_ids[i]],
            "confidence": round(float(confidences[i]), 3),
            "bbox_cxcywh": [round(float(x), 1) for x in boxes[i]]
        })
    
    return detections

# Decode the output from the inference above
detections = decode_yolo_output(output, conf_threshold=0.25)

print(f"Found {len(detections)} detections:")
for det in detections[:10]:  # Show first 10
    print(f"  {det['class']:15} conf={det['confidence']:.2f}  bbox={det['bbox_cxcywh']}")

---
## 2. BERT (Text Classification)

Input: `input_ids` and `attention_mask` tensors `[batch, seq_len]` (int64)  
Output: `logits` tensor `[batch, 2]` (float32)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

texts = [
    "I love this product!",
    "This is terrible.",
]

# Tokenize
encoded = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="np"
)

input_ids = encoded["input_ids"].astype(np.int64)
attention_mask = encoded["attention_mask"].astype(np.int64)

print(f"input_ids shape: {input_ids.shape}")
print(f"attention_mask shape: {attention_mask.shape}")

### BERT - gRPC and REST Binary

In [ ]:
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient

texts = [
    "I love this product!",
    "This is terrible.",
]

print(f"Texts: {texts}")

url = REST_URL.replace("http://", "").replace("https://", "")

# =============================================================================
# 1. gRPC Protocol
# =============================================================================
encoded = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="np")
input_ids_grpc = encoded["input_ids"].astype(np.int64).copy()
attention_mask_grpc = encoded["attention_mask"].astype(np.int64).copy()

client = grpcclient.InferenceServerClient(url=GRPC_URL)

inp_ids = grpcclient.InferInput("input_ids", list(input_ids_grpc.shape), "INT64")
inp_ids.set_data_from_numpy(input_ids_grpc)
inp_mask = grpcclient.InferInput("attention_mask", list(attention_mask_grpc.shape), "INT64")
inp_mask.set_data_from_numpy(attention_mask_grpc)
out = grpcclient.InferRequestedOutput("logits")

start = time.time()
resp = client.infer("bert-base-uncased", [inp_ids, inp_mask], outputs=[out], headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
logits_grpc = resp.as_numpy("logits")
client.close()

preds_grpc = ['NEGATIVE' if l[0] > l[1] else 'POSITIVE' for l in logits_grpc]
print(f"✓ gRPC:          {t_grpc:8.2f} ms | Predictions: {preds_grpc}")

# =============================================================================
# 2. REST (Binary) Protocol
# =============================================================================
encoded = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="np")
input_ids_binary = encoded["input_ids"].astype(np.int64).copy()
attention_mask_binary = encoded["attention_mask"].astype(np.int64).copy()

client = httpclient.InferenceServerClient(url=url)

inp_ids = httpclient.InferInput("input_ids", list(input_ids_binary.shape), "INT64")
inp_ids.set_data_from_numpy(input_ids_binary, binary_data=True)
inp_mask = httpclient.InferInput("attention_mask", list(attention_mask_binary.shape), "INT64")
inp_mask.set_data_from_numpy(attention_mask_binary, binary_data=True)
out = httpclient.InferRequestedOutput("logits", binary_data=True)

start = time.time()
resp = client.infer("bert-base-uncased", [inp_ids, inp_mask], outputs=[out], headers=http_headers)
t_binary = (time.time() - start) * 1000
logits_binary = resp.as_numpy("logits")

preds_binary = ['NEGATIVE' if l[0] > l[1] else 'POSITIVE' for l in logits_binary]
print(f"✓ REST (Binary): {t_binary:8.2f} ms | Predictions: {preds_binary}")

# Verify outputs match
print(f"\nOutputs match: {np.allclose(logits_grpc, logits_binary)}")

---
## 3. Whisper (Audio Transcription)

Input: `input_features` tensor `[1, 80, 3000]` (float32, mel spectrogram)  
Output: `transcription` (string)

In [ ]:
import librosa
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# Load audio file
audio_path = "../samples/audio_sample.wav"
audio, sr = librosa.load(audio_path, sr=16000, mono=True)

# Convert to mel spectrogram
inputs = processor(audio, sampling_rate=16000, return_tensors="np")
input_features = inputs.input_features.astype(np.float32)

print(f"Audio duration: {len(audio)/sr:.2f} seconds")
print(f"Input features shape: {input_features.shape}")
print(f"Input size: {input_features.nbytes / 1024:.1f} KB")

### Whisper - gRPC and REST Binary

In [ ]:
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient

# Load audio once
audio_path = "../samples/audio_sample.wav"
audio, sr = librosa.load(audio_path, sr=16000, mono=True)
print(f"Audio duration: {len(audio)/sr:.2f} seconds")

url = REST_URL.replace("http://", "").replace("https://", "")

# =============================================================================
# 1. gRPC Protocol
# =============================================================================
inputs_grpc = processor(audio, sampling_rate=16000, return_tensors="np")
features_grpc = inputs_grpc.input_features.astype(np.float32).copy()

client = grpcclient.InferenceServerClient(url=GRPC_URL)
inp = grpcclient.InferInput("input_features", list(features_grpc.shape), "FP32")
inp.set_data_from_numpy(features_grpc)
out = grpcclient.InferRequestedOutput("transcription")

start = time.time()
resp = client.infer("whisper-tiny-python", [inp], outputs=[out], headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
trans_grpc = resp.as_numpy("transcription")[0]
if isinstance(trans_grpc, bytes):
    trans_grpc = trans_grpc.decode("utf-8")
client.close()

print(f"✓ gRPC:          {t_grpc:8.2f} ms")

# =============================================================================
# 2. REST (Binary) Protocol
# =============================================================================
inputs_binary = processor(audio, sampling_rate=16000, return_tensors="np")
features_binary = inputs_binary.input_features.astype(np.float32).copy()

client = httpclient.InferenceServerClient(url=url)
inp = httpclient.InferInput("input_features", list(features_binary.shape), "FP32")
inp.set_data_from_numpy(features_binary, binary_data=True)
out = httpclient.InferRequestedOutput("transcription", binary_data=False)

start = time.time()
resp = client.infer("whisper-tiny-python", [inp], outputs=[out], headers=http_headers)
t_binary = (time.time() - start) * 1000
trans_binary = resp.as_numpy("transcription")[0]
if isinstance(trans_binary, bytes):
    trans_binary = trans_binary.decode("utf-8")

print(f"✓ REST (Binary): {t_binary:8.2f} ms")

print(f"\nTranscription: {trans_grpc}")

---
## 4. SmolLM-135M (Text Generation)

Input:
- `prompt`: string
- `max_tokens`: int32
- `temperature`: float32
- `top_p`: float32

Output:
- `generated_text`: string
- `token_count`: int32

### SmolLM - gRPC and REST Binary

In [ ]:
import tritonclient.grpc as grpcclient
import tritonclient.http as httpclient

prompt = "What is the capital of France?"
max_tokens = 128
temperature = 0.7
top_p = 0.9

print(f"Prompt: {prompt}")
print(f"Max tokens: {max_tokens}")

url = REST_URL.replace("http://", "").replace("https://", "")

# =============================================================================
# 1. gRPC Protocol
# =============================================================================
prompt_grpc = np.array([prompt], dtype=np.object_)
max_tokens_grpc = np.array([max_tokens], dtype=np.int32)
temperature_grpc = np.array([temperature], dtype=np.float32)
top_p_grpc = np.array([top_p], dtype=np.float32)

client = grpcclient.InferenceServerClient(url=GRPC_URL)

inp_prompt = grpcclient.InferInput("prompt", [1], "BYTES")
inp_prompt.set_data_from_numpy(prompt_grpc)
inp_max = grpcclient.InferInput("max_tokens", [1], "INT32")
inp_max.set_data_from_numpy(max_tokens_grpc)
inp_temp = grpcclient.InferInput("temperature", [1], "FP32")
inp_temp.set_data_from_numpy(temperature_grpc)
inp_top = grpcclient.InferInput("top_p", [1], "FP32")
inp_top.set_data_from_numpy(top_p_grpc)

outputs = [
    grpcclient.InferRequestedOutput("generated_text"),
    grpcclient.InferRequestedOutput("token_count"),
]

start = time.time()
resp = client.infer("smollm-135m-python", [inp_prompt, inp_max, inp_temp, inp_top],
                    outputs=outputs, headers=grpc_headers)
t_grpc = (time.time() - start) * 1000
text_grpc = resp.as_numpy("generated_text")[0]
if isinstance(text_grpc, bytes):
    text_grpc = text_grpc.decode("utf-8")
count_grpc = int(resp.as_numpy("token_count")[0])
client.close()

print(f"✓ gRPC:          {t_grpc:8.2f} ms | {count_grpc} tokens")

# =============================================================================
# 2. REST (Binary) Protocol
# =============================================================================
prompt_binary = np.array([prompt], dtype=np.object_)
max_tokens_binary = np.array([max_tokens], dtype=np.int32)
temperature_binary = np.array([temperature], dtype=np.float32)
top_p_binary = np.array([top_p], dtype=np.float32)

client = httpclient.InferenceServerClient(url=url)

inp_prompt = httpclient.InferInput("prompt", [1], "BYTES")
inp_prompt.set_data_from_numpy(prompt_binary, binary_data=False)  # BYTES always non-binary
inp_max = httpclient.InferInput("max_tokens", [1], "INT32")
inp_max.set_data_from_numpy(max_tokens_binary, binary_data=True)
inp_temp = httpclient.InferInput("temperature", [1], "FP32")
inp_temp.set_data_from_numpy(temperature_binary, binary_data=True)
inp_top = httpclient.InferInput("top_p", [1], "FP32")
inp_top.set_data_from_numpy(top_p_binary, binary_data=True)

outputs = [
    httpclient.InferRequestedOutput("generated_text", binary_data=False),
    httpclient.InferRequestedOutput("token_count", binary_data=True),
]

start = time.time()
resp = client.infer("smollm-135m-python", [inp_prompt, inp_max, inp_temp, inp_top],
                    outputs=outputs, headers=http_headers)
t_binary = (time.time() - start) * 1000
text_binary = resp.as_numpy("generated_text")[0]
if isinstance(text_binary, bytes):
    text_binary = text_binary.decode("utf-8")
count_binary = int(resp.as_numpy("token_count")[0])

print(f"✓ REST (Binary): {t_binary:8.2f} ms | {count_binary} tokens")

print(f"\nGenerated text:\n{text_grpc}")

---
## Summary

### Model Input/Output Reference

| Model | Input | Output |
|-------|-------|--------|
| yolov8n | `images` [B,3,640,640] FP32 | `output0` [B,84,8400] FP32 |
| bert-base-uncased | `input_ids`, `attention_mask` [B,128] INT64 | `logits` [B,2] FP32 |
| whisper-tiny-python | `input_features` [1,80,3000] FP32 | `transcription` BYTES |
| smollm-135m-python | `prompt` BYTES, `max_tokens` INT32, `temperature` FP32, `top_p` FP32 | `generated_text` BYTES, `token_count` INT32 |

### Protocol Comparison

| Protocol | Best For | `binary_data` | URL Format |
|----------|----------|---------------|------------|
| **gRPC** | All workloads, best performance | N/A (always binary) | `host:port` |
| **REST (Binary)** | HTTP infrastructure, good performance | `True` | `host:port` (strip http://) |
| **REST (JSON)** | Debugging only (see note below) | `False` | `host:port` (strip http://) |

### REST JSON Limitations

REST JSON (`binary_data=False`) has a known issue in Jupyter notebooks where it fails with JSON parsing errors when run after other cells in the same kernel session. This appears to be state corruption in the `tritonclient.http` library.

**REST JSON works in:**
- Standalone Python scripts (benchmarks work fine)
- Fresh Jupyter kernel with only the setup cell executed
- The sanity check cells (3-4) in this notebook

**Workaround:** Use gRPC or REST Binary for model inference cells. REST JSON is excluded from the main model cells due to this issue.

### Performance Notes

- **REST JSON** is significantly slower for large tensors (images, audio) due to ~10x larger payload size
- For YOLOv8n: REST JSON ~1400ms vs gRPC ~50ms (benchmark shows ~30x slower)
- Use gRPC or REST Binary for production workloads

### Key Takeaways

1. **Use gRPC** for best performance in all scenarios
2. **Use REST (Binary)** when you need HTTP compatibility
3. **REST (JSON)** has Jupyter compatibility issues - use standalone scripts if needed
4. **gRPC URL:** Just `host:port` (no protocol prefix)
5. **HTTP URL:** Strip `http://` prefix when creating InferenceServerClient
6. **String data:** Use `binary_data=False` for BYTES type inputs/outputs (strings)